# 🛠️ AWS SageMaker & Environment Setup — Amazon ML Challenge 2026

This notebook prepares, checks, and validates the AWS SageMaker execution environment:
1. **AWS Identity & Environment Detection** (Region, SageMaker execution role, S3 buckets)
2. **Hardware Diagnostics** (CUDA, GPU model, VRAM, CPU cores, RAM)
3. **Pinned Dependency Installation** (`requirements.txt`)
4. **Dataset Discovery & Path Verification** (Train/Test TSV files)
5. **Baseline Smoke Test** to confirm full environment readiness


In [ ]:
# 1. Environment & AWS Identity Detection
import os
import sys
import platform

print(f"Python Version: {platform.python_version()}")
print(f"OS Platform:    {platform.platform()}")

try:
    import sagemaker
    import boto3
    session = sagemaker.Session()
    role = sagemaker.get_execution_role()
    region = session.boto_region_name
    bucket = session.default_bucket()
    print(f"✅ AWS SageMaker Detected!")
    print(f"   Region: {region}")
    print(f"   Role:   {role}")
    print(f"   Bucket: {bucket}")
except Exception as e:
    print(f"ℹ️ Running in Local / Non-SageMaker environment: {e}")


In [ ]:
# 2. Hardware Diagnostics (GPU / CPU / Memory)
import psutil
import torch

print(f"CPU Physical Cores: {psutil.cpu_count(logical=False)}")
print(f"CPU Logical Cores:  {psutil.cpu_count(logical=True)}")
print(f"Total RAM:          {psutil.virtual_memory().total / (1024**3):.2f} GB")
print(f"Available RAM:      {psutil.virtual_memory().available / (1024**3):.2f} GB")

cuda_available = torch.cuda.is_available()
print(f"\nCUDA Available:     {cuda_available}")
if cuda_available:
    print(f"GPU Device:         {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory:         {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("Running in CPU mode. GBDT and multi-pass inverted indexing are fully CPU-optimized.")


In [ ]:
# 3. Install Pinned Competition Dependencies
!pip install -r ../requirements.txt -q
!pip install lightgbm scikit-learn pandas numpy pyyaml rich tqdm -q
print("✅ Dependencies successfully installed and verified.")


In [ ]:
# 4. Dataset Discovery & Path Verification
from pathlib import Path

REPO_ROOT = Path("..").resolve()
TRAIN_DIR = REPO_ROOT / "resources" / "student_resource" / "dataset" / "train"
TEST_DIR = REPO_ROOT / "resources" / "student_resource" / "dataset" / "test"

expected_train_files = ["train_source1.tsv", "train_source2.tsv", "train_source3.tsv", "train_ground_truth.tsv"]
expected_test_files = ["test_source1.tsv", "test_source2.tsv", "test_source3.tsv"]

print(f"Checking Train Directory: {TRAIN_DIR}")
for f in expected_train_files:
    p = TRAIN_DIR / f
    status = f"✅ {p.stat().st_size / (1024**2):.1f} MB" if p.exists() else "❌ MISSING"
    print(f"  {f:25s}: {status}")

print(f"\nChecking Test Directory: {TEST_DIR}")
for f in expected_test_files:
    p = TEST_DIR / f
    status = f"✅ {p.stat().st_size / (1024**2):.1f} MB" if p.exists() else "❌ MISSING"
    print(f"  {f:25s}: {status}")


In [ ]:
# 5. Quick Pipeline Smoke Test
!python ../scripts/run_pipeline.py --sample --n-s1 200
print("✅ Environment is fully verified and ready for experiment runs!")
